## Export clipped data to 
Author: Lara Spearman <br>
Date: 01/09/2025 <br>
Purpose: Takes output csv from data prep.ipynb and exports it as html file

In [57]:
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
pd.options.display.float_format = '{:,.1f}'.format
import plotly.express as px
from pathlib import Path
import os

In [58]:
pc = "Lydiard Millicent CP"

# Get current working directory
cwd = Path.cwd()
location_data = "C:/Users/LaraS/Wiltshire Wildlife Trust/WSBRC Data - Documents/Services/Neighbourhood_Planning/_NeigbourhoodPlansPython/NeigbourhoodPlansPython.gdb"
output_folder = f"{cwd}/../Jobs/"

In [59]:
gdf = gpd.read_file(location_data, layer= "OS_BL_Parish_Wilts")

In [60]:
# Create rank for PC area
gdf['AreaPC'] = gdf['Hectares']
df_pc_sorted = gdf.sort_values(by = "AreaPC", ascending=False)
df_pc_sorted['rankArea'] = list(range(1, df_pc_sorted.shape[0]+1))

In [61]:
dfpc = df_pc_sorted[df_pc_sorted.Name == pc]
pcAreaRank =dfpc['rankArea'].item()

In [62]:
# Filter datasets to pc
df = pd.read_csv("data/outputClippedData.csv")
df_pc = df[df.Name == pc]
pcAreaHa = df_pc['AreaPC'].iloc[0].astype(int)

In [63]:
# Select only items ranked higher than 10 for highlights boxes
df_top_ten =  df[df['rank']<=10]
df_top_ten.to_csv("data/topTen.csv", index=False)

In [64]:
# Select only items ranked higher than 10 for highlights boxes
df_pc_sorted = df_pc.sort_values(by = "rank", ascending=True)
df_pc_filter = df_pc_sorted[df_pc_sorted['rank']<=10]

# Select datasets you want to show in highlights boxes
df_pc_filter = df_pc_filter[df_pc_filter.datasetName.isin(['Ancient Woodland Inventory','National Forest Inventory','Greenspace','PHI','WSBRC Priority Habitats'])]


In [68]:
df_pc = df_pc.sort_values("mapGroup")

In [69]:
# Create dictionary to store all information for output
table_dict = {}
for category in df_pc.mapGroup:
    dataset_dict = {}
    df_pc_category =  df_pc[df_pc.mapGroup == category]
    
    for dataset in df_pc_category.datasetName:
        if df_pc_category.shape[0] >0:
            print(dataset)
            df_pc_filter_dataset = df_pc_category[df_pc_category.datasetName == dataset]
            # Check if dataset is an area, and if it is, include the value percent
            if df_pc_filter_dataset['unit'].iloc[0] == "Ha":
                df_small = df_pc_filter_dataset[['groupColumnValue','value', 'valuePercent']]
                
                df_small = df_small.rename(columns = {'groupColumnValue':df_pc_filter_dataset['groupColumnName'].iloc[0],
                                                    "value":df_pc_filter_dataset['unit'].iloc[0],
                                                    "valuePercent":"Percentage of PC"})
            # If dataset in metres or count, do not include valuePercent as is not relevant
            elif df_pc_filter_dataset['unitName'].iloc[0].lower() == "length" or df_pc_filter_dataset['unitName'].iloc[0].lower() == "count" :
                df_small = df_pc_filter_dataset[['groupColumnValue','value']]
                
                df_small = df_small.rename(columns = {'groupColumnValue':df_pc_filter_dataset['groupColumnName'].iloc[0],
                                                    "value":df_pc_filter_dataset['unit'].iloc[0]})
                
            # Check if column value has only one item, and if so, do not include figure as no point showing on graph
            if df_small[df_pc_filter_dataset['groupColumnName'].iloc[0]].unique().shape[0]==1:
                if df_pc_filter_dataset['groupColumnName'].iloc[0]== "Total":
                    dataset_dict[dataset] = {"table":"",
                                        "fig":"",
                                        "text": f"There is a total of {df_pc_filter_dataset['value'].values[0]} {df_pc_filter_dataset['unit'].values[0]} ",
                                        }
                else:
                    dataset_dict[dataset] = {"table":"",
                                        "fig":"",
                                        "text":f"There is {df_pc_filter_dataset['value'].values[0]} {df_pc_filter_dataset['unit'].values[0]} of  {df_pc_filter_dataset['groupColumnValue'].values[0]}"
                                        }
            # If column value as more than one item, add in figure. 
            else: 
                df_small = df_small.sort_values(by = df_pc_filter_dataset['unit'].iloc[0] , ascending=False)
                fig = px.bar(df_small, x = df_pc_filter_dataset['groupColumnName'].iloc[0],y =df_pc_filter_dataset['value'], template="plotly_white", height = 350 )
                fig.update_layout(
                    margin={'t':0,'l':0,'b':0,'r':0}
                )
                dataset_dict[dataset] = {"table":df_small.to_html(index = False),
                                     "fig":""#fig.to_html(full_html=False)
                                     }
    table_dict[category] = dataset_dict

Public Rights of Way
Country Parks
CRoW Access Land
Public Rights of Way
Public Rights of Way
Country Parks
CRoW Access Land
Public Rights of Way
Public Rights of Way
Country Parks
CRoW Access Land
Public Rights of Way
Public Rights of Way
Country Parks
CRoW Access Land
Public Rights of Way
Countryside Stewardship Scheme
Environmental Stewardship Scheme Agreements
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Environmental Stewardship Scheme Agreements
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Environmental Stewardship Scheme Agreements
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Environmental Stewardship Scheme Agreements
Countryside Stewardship Scheme
Countryside Stewardship Scheme
Agricultural Land Classification
Agricultural Land Classification
Agricultural Land Classification
Agricultural Land Classification
Bedrock
Bedrock
Bedrock
Bedr

In [70]:
environment = Environment(loader=FileSystemLoader("templates/"))
template = environment.get_template("Stats template.html")

#Create folder if not already there
if not os.path.exists(f"{output_folder}{pc}"):
    os.makedirs(f"{output_folder}{pc}")

filename = f"{output_folder}{pc}/{pc}_stats.html"
# filename = f"outputReports/test_stats.html"

content = template.render(
    pc = pc,
    pcAreaHa = format(pcAreaHa.round(-1),","),
    pcAreaRank = pcAreaRank,
    scores = df_pc_filter,
    data = table_dict

)
with open(filename, mode="w", encoding="utf-8") as message:
    message.write(content)
    print(f"... wrote {filename}")

... wrote c:\Users\LaraS\Wiltshire Wildlife Trust\WSBRC Data - Documents\Services\Neighbourhood_Planning\Analysis\Summary stats\neighbourhood-plan/../Jobs/Lydiard Millicent CP/Lydiard Millicent CP_stats.html
